# Tutorial: MedSAM para segmentación pulmonar

Este notebook explica **MedSAM** paso a paso: qué es, cómo está construido, qué necesita para segmentar, cómo se usa y qué cuidar. Se aplica primero al dataset Montgomery + Shenzhen, que tiene máscaras y permite medir el Dice, y después a TBX11K, que no las tiene.

A diferencia del tutorial de U-Net, aquí **no se entrena ninguna red**: MedSAM ya viene entrenado y se usa tal cual. Lo que hay que diseñar es el **prompt**: la caja que le indica al modelo qué estructura segmentar.

**Contenido**
1. ¿Qué es MedSAM?
2. ¿Hacen falta máscaras para usar MedSAM?
3. Instalación y carga del modelo
4. Arquitectura de MedSAM, pieza por pieza
5. Los datos y el preprocesamiento
6. El prompt: una caja por pulmón
7. Inferencia paso a paso
8. Evaluación en el conjunto de prueba
9. Sensibilidad a la caja y cajas automáticas
10. Aplicación a TBX11K
11. Qué hay que tener en cuenta
12. Ajuste fino (*fine-tuning*)
13. Relación con el proyecto

**Requisitos**
- Entorno instalado (`pip install -r requirements.txt` y `pip install -e .`).
- Dataset preparado: `python -m tbseg.segmentation.prepare`.
- Dos paquetes que no están en `requirements.txt` (sección 3):
  ```bash
  pip install torchvision==0.29.0
  pip install git+https://github.com/facebookresearch/segment-anything.git
  ```
- Pesos de MedSAM (375 MB). La sección 3 los descarga a `models/medsam/`.
- GPU recomendada. Medido en una GTX 1650 Ti (4 GB): entre 0,8 y 3,4 s por imagen según la ejecución, y 2,8 GB de memoria de GPU. En CPU (4 hilos): ≈17 s por imagen, así que la sección 8 (212 imágenes) tarda cerca de 1 hora.

> **Advertencia (sección 5):** MedSAM se desarrolló con este mismo dataset de Montgomery + Shenzhen. El Dice que se obtiene aquí es optimista y **no sirve para comparar MedSAM con U-Net**.

### 0. Configuración

In [ ]:
import time
import urllib.request

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from matplotlib.patches import FancyArrowPatch, Rectangle
from segment_anything import sam_model_registry   # sección 3
from tqdm.auto import tqdm

from tbseg.config import IMAGE_SIZE, MODELS_DIR, SEED, SEGTRAIN_CSV, SEGTRAIN_DIR
from tbseg.data import load_gray, load_subset

MEDSAM_SIZE = 1024   # resolución de entrada fija de MedSAM
MEDSAM_CKPT = MODELS_DIR / "medsam" / "medsam_vit_b.pth"

np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", DEVICE, f"({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else "")

# Colores: azul = máscara real, naranja = predicción, verde = caja (prompt)
C_TRUE, C_PRED, C_BOX = "#2a78d6", "#eb6834", "#2e9e5b"
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})

## 1. ¿Qué es MedSAM?

**SAM** (*Segment Anything Model*, Kirillov et al., 2023) es un modelo de Meta AI entrenado con imágenes naturales para segmentar un objeto a partir de un **prompt**: uno o varios puntos, una caja o una máscara aproximada. No sabe de antemano qué es un pulmón: segmenta lo que el prompt le indica.

**MedSAM** (Ma et al., 2024, *Nature Communications* 15: 654) es SAM ajustado para imagen médica:

| | |
|---|---|
| Datos de desarrollo | 1 570 263 pares imagen-máscara, 10 modalidades (entre ellas radiografía de tórax), más de 30 tipos de cáncer |
| Prompt | **Solo cajas** |
| Qué se entrenó | Codificador de imagen y decodificador de máscara; el codificador de prompt quedó fijo |
| Pérdida | Dice + entropía cruzada, sin ponderar (la misma combinación del tutorial de U-Net) |
| Optimización | AdamW, tasa de aprendizaje 1e-4, *weight decay* 0,01, 150 épocas, 20 GPU A100 de 80 GB |

Por qué solo cajas, según el artículo: la caja da un contexto espacial sin ambigüedad sobre la región de interés, mientras que un punto puede ser ambiguo cuando hay estructuras vecinas parecidas.

**U-Net frente a MedSAM**

| | U-Net (tutorial anterior) | MedSAM |
|---|---|---|
| Entrenamiento | Desde cero, con 492 radiografías de Montgomery + Shenzhen | Ya entrenado; en este tutorial no se entrena |
| Entrada | Radiografía | Radiografía **+ una caja por estructura** |
| Qué segmenta | Lo que aprendió: pulmón | Lo que indique la caja |
| Parámetros | ≈1,9 M (versión reducida) | ≈93,7 M |
| Resolución de entrada | 256 (tutorial) / 512 (proyecto) | 1024, fija |
| ¿Necesita máscaras? | Sí, para entrenar | No para segmentar (sección 2) |

## 2. ¿Hacen falta máscaras para usar MedSAM?

**Para segmentar, no.** MedSAM recibe una radiografía y una caja por estructura, y devuelve la máscara. Como no se entrena, no necesita pares *(radiografía, máscara)*.

**Lo que sí necesita es la caja.** TBX11K no tiene cajas de pulmón (sus anotaciones son cajas de lesiones de TB), así que hay que obtenerlas de alguna fuente:

| Fuente de la caja | ¿Necesita máscaras? | Uso |
|---|---|---|
| Dibujada a mano | No | Viable para pocas imágenes; no para 1600 radiografías × 2 pulmones |
| Calculada desde la máscara real (**caja oráculo**) | Sí, de cada imagen | Solo para evaluar: mide el mejor caso posible |
| Caja típica aprendida de un dataset con máscaras (**caja a priori**) | Sí, de otro dataset (Montgomery + Shenzhen) | Automática; se puede aplicar a TBX11K |
| Calculada desde la máscara de otro método (p. ej. U-Net) | No adicionales | Automática; MedSAM refina la máscara de otro método |

En este tutorial las máscaras de Montgomery + Shenzhen se usan para tres cosas: (1) calcular el Dice, (2) construir la caja a priori y (3) elegir los márgenes de la caja en validación. El ajuste fino también las usaría, pero aquí no se hace (sección 12).

**Qué hace falta para empezar a usar MedSAM**
1. Los paquetes `torchvision` y `segment-anything` (sección 3).
2. Los pesos `medsam_vit_b.pth` (sección 3).
3. El mismo preprocesamiento con el que se entrenó (sección 5).
4. Una fuente de cajas, una por pulmón (secciones 6 y 9).

## 3. Instalación y carga del modelo

### 3.1 Paquetes

MedSAM no se publica en PyPI. Su repositorio indica que los pesos se cargan con el cargador de SAM, así que basta con el paquete `segment_anything` de Meta:

```bash
pip install torchvision==0.29.0
pip install git+https://github.com/facebookresearch/segment-anything.git
```

- `segment_anything` importa `torchvision` al cargarse. La versión 0.29.0 es la que declara `torch==2.14.0`, la versión fijada en `requirements.txt`.
- No se instala el repositorio de MedSAM (`pip install -e .` dentro de `MedSAM/`): su `setup.py` instala `monai` con restricciones sobre paquetes `nvidia-*-cu12`, que pueden chocar con el `torch` (CUDA 13.0) del proyecto. El código de inferencia de MedSAM es una función corta (`medsam_inference` en `MedSAM_Inference.py`) que se reproduce en la sección 7.

### 3.2 Pesos

El archivo oficial `medsam_vit_b.pth` está en la [carpeta de Google Drive](https://drive.google.com/drive/folders/1ETWmi4AiniJeWOt6HAsYgTjYv_fkgzoN) que enlaza el repositorio de MedSAM. La celda lo descarga a `models/medsam/` (carpeta no versionada) si no existe y comprueba el tamaño. Si la descarga falla, descárgalo a mano desde ese enlace y colócalo en la misma ruta.

In [ ]:
MEDSAM_URL = "https://drive.usercontent.google.com/download?id=1UAmWL88roYR7wKlnApw5Bcuzf2iQgk6_&export=download&confirm=t"
MEDSAM_BYTES = 375_049_145

if not MEDSAM_CKPT.exists():
    MEDSAM_CKPT.parent.mkdir(parents=True, exist_ok=True)
    print(f"Descargando pesos de MedSAM en {MEDSAM_CKPT} ...")
    urllib.request.urlretrieve(MEDSAM_URL, MEDSAM_CKPT)
if MEDSAM_CKPT.stat().st_size != MEDSAM_BYTES:
    raise RuntimeError(
        f"{MEDSAM_CKPT} pesa {MEDSAM_CKPT.stat().st_size} bytes y se esperaban {MEDSAM_BYTES}. "
        "Bórralo y descárgalo a mano desde https://drive.google.com/drive/folders/1ETWmi4AiniJeWOt6HAsYgTjYv_fkgzoN"
    )
print(f"Pesos listos: {MEDSAM_CKPT.name} ({MEDSAM_CKPT.stat().st_size / 1e6:.0f} MB)")

### 3.3 Cargar el modelo

1. `sam_model_registry["vit_b"]()` construye la arquitectura de SAM con el codificador ViT-B y pesos aleatorios.
2. `load_state_dict` reemplaza esos pesos por los de MedSAM. Debe responder `All keys matched successfully`: todas las capas encontraron su peso.

Los pesos están guardados como tensores de GPU, por eso se cargan con `map_location="cpu"`: así también funciona en un equipo sin GPU. Por la misma razón no se usa `sam_model_registry["vit_b"](checkpoint=...)`, que en el paquete de Meta llama a `torch.load` sin `map_location`.

In [ ]:
medsam = sam_model_registry["vit_b"]()
print(medsam.load_state_dict(torch.load(MEDSAM_CKPT, map_location="cpu")))
medsam = medsam.to(DEVICE).eval()   # eval(): modo inferencia

for name in ["image_encoder", "prompt_encoder", "mask_decoder"]:
    print(f"{name:15s} {sum(p.numel() for p in getattr(medsam, name).parameters()):>12,} parámetros")

## 4. Arquitectura de MedSAM, pieza por pieza

MedSAM conserva la arquitectura de SAM. Tiene tres partes:

1. **Codificador de imagen** (ViT-B, 89,7 M parámetros). Divide la imagen de 1024×1024 en parches de 16×16 (64×64 = 4096 parches) y los procesa con 12 capas de *transformer*. Produce un **embedding** de 256×64×64: una representación de la imagen con 256 valores por cada parche. Es la parte pesada y **se ejecuta una vez por imagen**.
2. **Codificador de prompt** (6 220 parámetros). Convierte cada caja (dos esquinas) en 2 vectores de 256 valores (*tokens*). Como no recibe máscara de entrada, agrega además un embedding denso fijo de 256×64×64.
3. **Decodificador de máscara** (4,1 M parámetros). Un *transformer* liviano que cruza los tokens de la caja con el embedding de la imagen y produce **logits de 256×256**. Se reescalan al tamaño de la imagen y se pasan por la sigmoide y un umbral, igual que en U-Net.

Consecuencia práctica: el embedding se calcula una vez y se reutiliza con todas las cajas que se quieran probar. Probar otra caja solo cuesta el decodificador. Este tutorial lo aprovecha: calcula los embeddings de validación y prueba una vez (sección 8) y después prueba muchas cajas (sección 9).

In [ ]:
def draw_medsam():
    fig, ax = plt.subplots(figsize=(12, 4.2))

    def block(x, y, w, text, fc, ec):
        ax.add_patch(Rectangle((x, y), w, 1, fc=fc, ec=ec, lw=1.5))
        ax.text(x + w / 2, y + 0.5, text, ha="center", va="center", fontsize=9)

    def arrow(x0, y0, x1, y1):
        ax.add_patch(FancyArrowPatch((x0, y0), (x1, y1), arrowstyle="-|>", mutation_scale=12, color="#52514e", lw=1.5))

    gray = ("#f1f0ec", "#8a8984")
    block(0, 2, 2, "Radiografía\n1024×1024×3", *gray)
    block(2.8, 2, 3, "Codificador de imagen\nViT-B · 89,7 M", "#dbe8f8", C_TRUE)
    block(6.6, 2, 2, "Embedding\n256×64×64", *gray)
    block(0, 0, 2, "Caja\n[x0, y0, x1, y1]", "#dcf1e3", C_BOX)
    block(2.8, 0, 3, "Codificador de prompt\n6 220", "#dcf1e3", C_BOX)
    block(6.6, 0, 2, "Tokens\n2×256", *gray)
    block(9.4, 1, 3, "Decodificador\nde máscara · 4,1 M", "#fde3d7", C_PRED)
    block(13.2, 1, 2, "Logits\n256×256", *gray)
    block(16.2, 1, 2, "Máscara\n512×512", *gray)
    arrow(2, 2.5, 2.8, 2.5); arrow(5.8, 2.5, 6.6, 2.5); arrow(8.6, 2.5, 9.4, 1.7)
    arrow(2, 0.5, 2.8, 0.5); arrow(5.8, 0.5, 6.6, 0.5); arrow(8.6, 0.5, 9.4, 1.3)
    arrow(12.4, 1.5, 13.2, 1.5); arrow(15.2, 1.5, 16.2, 1.5)
    ax.text(15.7, 2.15, "sigmoide +\nreescalado", ha="center", fontsize=8, color="#52514e")
    ax.text(4.3, 3.25, "una vez por imagen (pesado)", ha="center", fontsize=9, color=C_TRUE)
    ax.text(7.9, -0.45, "una vez por caja (liviano)", ha="center", fontsize=9, color=C_BOX)
    ax.set_xlim(-0.3, 18.5); ax.set_ylim(-0.8, 3.6); ax.axis("off")
    ax.set_title("MedSAM: flujo de inferencia", loc="left")
    plt.show()

draw_medsam()

## 5. Los datos y el preprocesamiento

Se usa el mismo dataset del tutorial de U-Net: Montgomery + Shenzhen, 704 radiografías con máscara manual de pulmón, ya llevadas a 512×512 por `python -m tbseg.segmentation.prepare` y divididas en train/val/test (70/15/15).

### 5.1 Advertencia: MedSAM ya vio estas imágenes

La Tabla 3 del material suplementario de MedSAM lista **"Chest Xray Masks and Labels"** (Kaggle, `nikhilpandey360/chest-xray-masks-and-labels`), 704 imágenes, objetivo "Lung": es el mismo dataset y la misma descarga del paso 7 del README. Está entre los datasets de validación interna, y los autores dividieron todos sus datos al azar en 80 % entrenamiento, 10 % ajuste y 10 % validación. Por lo tanto, se espera que cerca del 80 % de estas radiografías haya estado en el entrenamiento de MedSAM. No encontré en el artículo ni en el suplemento la lista de qué imágenes quedaron en cada parte.

Consecuencias:
- **El Dice de este tutorial es optimista.** Es una fuga de información como la descrita en la sección 9 del tutorial de U-Net, pero del lado del modelo.
- **No sirve para comparar MedSAM con U-Net.** U-Net nunca vio el conjunto de prueba; MedSAM probablemente sí vio parte. Una comparación justa necesita imágenes que ninguno de los dos haya visto: la revisión visual en TBX11K o un dataset con máscaras de pulmón que no esté en la Tabla 3 del suplemento.
- **Sí sirve para aprender a usar el modelo** y para comparar estrategias de caja entre sí, con la misma reserva.

### 5.2 Preprocesamiento

MedSAM espera la imagen en el mismo formato con el que se entrenó. Según el artículo y el código del repositorio:

1. **Recorte de intensidades**: para rayos X, el artículo recorta los valores a los percentiles 0,5 y 99,5 antes de reescalar.
2. **1024×1024**: tamaño fijo del codificador de imagen. Las radiografías de 512 se amplían con interpolación cúbica.
3. **Valores en [0, 1]** por min-max (`MedSAM_Inference.py`).
4. **3 canales**: la radiografía en gris se repite en los tres canales RGB.

> **No usar `SamPredictor` ni la normalización de SAM.** SAM normaliza con la media y la desviación de ImageNet (`pixel_mean`, `pixel_std`). MedSAM se entrenó pasando la imagen en [0, 1] directamente al codificador de imagen (`train_one_gpu.py` lo verifica con un `assert`). Por eso aquí se llama a `medsam.image_encoder` sin pasar por `SamPredictor`.

In [ ]:
split = pd.read_csv(SEGTRAIN_CSV)
display(pd.crosstab(split["source"], split["split"], margins=True, margins_name="total"))


def load_pair(image_id):
    """Imagen en gris (uint8, 512x512) y máscara booleana del mismo tamaño."""
    img = cv2.imread(str(SEGTRAIN_DIR / "images" / f"{image_id}.png"), cv2.IMREAD_GRAYSCALE)
    mask = cv2.imread(str(SEGTRAIN_DIR / "masks" / f"{image_id}.png"), cv2.IMREAD_GRAYSCALE)
    return img, mask >= 128


def preprocess(img):
    """Gris uint8 (H, W) -> tensor (1, 3, 1024, 1024) en [0, 1], como en el entrenamiento de MedSAM."""
    lo, hi = np.percentile(img, (0.5, 99.5))
    img = np.clip((img.astype(np.float32) - lo) / max(hi - lo, 1e-8), 0, 1)
    img = cv2.resize(img, (MEDSAM_SIZE, MEDSAM_SIZE), interpolation=cv2.INTER_CUBIC)
    img = (img - img.min()) / max(img.max() - img.min(), 1e-8)   # la interpolación cúbica puede salir de [0, 1]
    return torch.from_numpy(img.astype(np.float32))[None, None].repeat(1, 3, 1, 1)


def draw(ax, img, true=None, pred=None, boxes=(), title="", box_style="-"):
    """Radiografía con contornos (azul: real, naranja: predicción) y cajas (verde)."""
    ax.imshow(img, cmap="gray")
    for m, color in [(true, C_TRUE), (pred, C_PRED)]:
        if m is not None and m.any():
            ax.contour(m.astype(np.uint8), levels=[0.5], colors=color, linewidths=1.5)
    for x0, y0, x1, y1 in boxes:
        ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec=C_BOX, lw=1.5, ls=box_style))
    ax.set_title(title, fontsize=9)
    ax.axis("off")


ids = split["image_id"].sample(4, random_state=SEED).tolist()
fig, axes = plt.subplots(2, 4, figsize=(11, 5.8))
for c, image_id in enumerate(ids):
    img, mask = load_pair(image_id)
    x = preprocess(img)
    draw(axes[0, c], img, true=mask, title=f"{image_id}\n{img.shape[0]}×{img.shape[1]}, valores {img.min()}–{img.max()}")
    draw(axes[1, c], x[0, 0].numpy(), title=f"{x.shape[2]}×{x.shape[3]}×{x.shape[1]}, valores {x.min():.0f}–{x.max():.0f}")
axes[0, 0].text(-40, 256, "Original", rotation=90, va="center", ha="right", fontsize=10)
axes[1, 0].text(-80, 512, "Entrada a MedSAM", rotation=90, va="center", ha="right", fontsize=10)
fig.tight_layout()
plt.show()

## 6. El prompt: una caja por pulmón

Una caja se escribe como `[x0, y0, x1, y1]`: esquina superior izquierda y esquina inferior derecha, en píxeles.

**MedSAM segmenta una estructura por caja.** Los pulmones son dos estructuras separadas por el mediastino. Una sola caja que cubra ambos incluye también el corazón y el mediastino, y el modelo debe decidir si segmenta los dos pulmones o todo el contenido de la caja. El artículo reporta los resultados de radiografía de tórax por separado para "Left Lung" y "Right Lung" (tablas suplementarias), lo que corresponde a una caja por pulmón. La sección 8 compara ambas opciones.

Para obtener la caja oráculo de cada pulmón desde la máscara real:
1. Se buscan los **componentes conectados** de la máscara (regiones de píxeles de pulmón que se tocan) y se toman los dos más grandes.
2. La caja de cada componente va del píxel más a la izquierda al más a la derecha, y del más alto al más bajo.
3. Se ordenan de izquierda a derecha **en la imagen**. En una radiografía posteroanterior, el pulmón derecho del paciente aparece a la izquierda de la imagen.

MedSAM entrega una máscara por caja. La máscara final es la **unión** de las dos.

In [ ]:
def lung_boxes(mask):
    """Una caja [x0, y0, x1, y1] por pulmón, ordenadas de izquierda a derecha en la imagen.

    Usa los dos componentes conectados más grandes. Si los pulmones forman un solo
    componente, lo divide por la columna central.
    """
    _, labels, stats, _ = cv2.connectedComponentsWithStats(mask.astype(np.uint8), connectivity=8)
    largest = np.argsort(stats[1:, cv2.CC_STAT_AREA])[::-1][:2] + 1   # la etiqueta 0 es el fondo
    if len(largest) == 2:
        parts = [labels == i for i in largest]
    else:
        mid = mask.shape[1] // 2
        parts = [mask.copy(), mask.copy()]
        parts[0][:, mid:] = False
        parts[1][:, :mid] = False
    boxes = []
    for part in parts:
        ys, xs = np.nonzero(part)
        boxes.append([xs.min(), ys.min(), xs.max(), ys.max()])
    return np.array(sorted(boxes))


def full_box(mask):
    """Una sola caja que cubre ambos pulmones."""
    ys, xs = np.nonzero(mask)
    return np.array([[xs.min(), ys.min(), xs.max(), ys.max()]])


def expand(boxes, margin, size=IMAGE_SIZE):
    """Agranda cada caja `margin` píxeles por lado, sin salir de la imagen."""
    return np.clip(np.asarray(boxes) + np.array([-margin, -margin, margin, margin]), 0, size - 1)


fig, axes = plt.subplots(1, 4, figsize=(12, 3.6))
for ax, image_id in zip(axes, ids):
    img, mask = load_pair(image_id)
    draw(ax, img, true=mask, boxes=lung_boxes(mask), title=image_id)
    x0, y0, x1, y1 = full_box(mask)[0]
    ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec=C_BOX, lw=1, ls="--"))
fig.suptitle("Continua: una caja por pulmón (oráculo) — discontinua: una caja para ambos", x=0.01, ha="left")
fig.tight_layout()
plt.show()

## 7. Inferencia paso a paso

Se recorre una imagen por las tres partes del modelo, imprimiendo la forma del tensor en cada paso. El código reproduce `medsam_inference` de `MedSAM_Inference.py`.

- Las cajas se definen en píxeles de la imagen de 512 y se **reescalan a 1024**, el espacio de coordenadas del modelo.
- Las dos cajas se procesan juntas: el decodificador repite el embedding de la imagen una vez por caja.
- `multimask_output=False`: SAM puede devolver 3 máscaras candidatas cuando el prompt es ambiguo; MedSAM devuelve una.
- El decodificador también devuelve una estimación de la calidad de la máscara (IoU predicho). MedSAM no la usa y su entrenamiento no incluyó una pérdida para esa salida, así que no conviene usarla como control de calidad.
- La primera llamada al codificador de imagen tarda más: incluye la inicialización de la GPU.

In [ ]:
image_id = ids[0]
img, mask = load_pair(image_id)
boxes = lung_boxes(mask)
x = preprocess(img).to(DEVICE)
print("Entrada al codificador de imagen:", tuple(x.shape))

with torch.no_grad():
    start = time.time()
    embedding = medsam.image_encoder(x)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    print(f"Embedding de la imagen:          {tuple(embedding.shape)} ({time.time() - start:.2f} s)")

    box_1024 = torch.as_tensor(boxes, dtype=torch.float, device=DEVICE) * (MEDSAM_SIZE / IMAGE_SIZE)
    sparse, dense = medsam.prompt_encoder(points=None, boxes=box_1024[:, None, :], masks=None)
    print("Cajas en 512:", boxes.tolist(), "-> en 1024:", box_1024.int().tolist())
    print(f"Tokens de las cajas:             {tuple(sparse.shape)}   (cajas, tokens, dimensión)")
    print(f"Embedding denso del prompt:      {tuple(dense.shape)}")

    low_res, iou_pred = medsam.mask_decoder(
        image_embeddings=embedding,
        image_pe=medsam.prompt_encoder.get_dense_pe(),   # codificación de la posición de cada parche
        sparse_prompt_embeddings=sparse,
        dense_prompt_embeddings=dense,
        multimask_output=False,
    )
    print(f"Logits del decodificador:        {tuple(low_res.shape)}")
    probs = F.interpolate(torch.sigmoid(low_res), size=img.shape, mode="bilinear", align_corners=False)
    probs = probs[:, 0].cpu().numpy()
    print(f"Probabilidad reescalada:         {probs.shape}")

El resultado es un **mapa de probabilidad por caja**. Se une con un umbral de 0,5, igual que en U-Net, y se compara con la máscara real mediante el Dice (definido en la sección 5 del tutorial de U-Net).

In [ ]:
def to_mask(probs, threshold=0.5):
    """Unión de las máscaras de todas las cajas: un píxel es pulmón si alguna caja lo marca."""
    return (probs > threshold).any(axis=0)


def dice(pred, true):
    inter = np.logical_and(pred, true).sum()
    total = pred.sum() + true.sum()
    return 2 * inter / total if total else 1.0


pred = to_mask(probs)
fig, axes = plt.subplots(1, 4, figsize=(14, 3.8), layout="constrained")
draw(axes[0], img, boxes=boxes, title="Radiografía y cajas (prompt)")
for k in range(2):
    im = axes[k + 1].imshow(probs[k], cmap="Blues", vmin=0, vmax=1)
    axes[k + 1].set_title(f"Probabilidad de pulmón, caja {k + 1}", fontsize=9)
    axes[k + 1].axis("off")
fig.colorbar(im, ax=axes[1:3], fraction=0.03)
draw(axes[3], img, true=mask, pred=pred, title=f"Unión (naranja) vs. real (azul) — Dice {dice(pred, mask):.3f}")
plt.show()

Los pasos anteriores se agrupan en dos funciones:
- `embed(img)`: codificador de imagen, una vez por radiografía. El embedding se guarda en CPU para reutilizarlo.
- `segment(embedding, boxes)`: codificador de prompt + decodificador de máscara, una vez por conjunto de cajas.

In [ ]:
@torch.no_grad()
def embed(img):
    """Embedding (1, 256, 64, 64) de una radiografía en gris, en CPU."""
    return medsam.image_encoder(preprocess(img).to(DEVICE)).cpu()


@torch.no_grad()
def segment(embedding, boxes, size=IMAGE_SIZE):
    """Probabilidad de pulmón (K, size, size), una por caja. boxes: (K, 4) en píxeles de la imagen size x size."""
    box_1024 = torch.as_tensor(np.asarray(boxes), dtype=torch.float, device=DEVICE) * (MEDSAM_SIZE / size)
    sparse, dense = medsam.prompt_encoder(points=None, boxes=box_1024[:, None, :], masks=None)
    low_res, _ = medsam.mask_decoder(
        image_embeddings=embedding.to(DEVICE),
        image_pe=medsam.prompt_encoder.get_dense_pe(),
        sparse_prompt_embeddings=sparse,
        dense_prompt_embeddings=dense,
        multimask_output=False,
    )
    probs = F.interpolate(torch.sigmoid(low_res), size=(size, size), mode="bilinear", align_corners=False)
    return probs[:, 0].cpu().numpy()


assert np.allclose(segment(embed(img), boxes), probs, atol=1e-4)

## 8. Evaluación en el conjunto de prueba

Se calculan una vez los embeddings de validación y prueba (212 radiografías) y se guardan en memoria (≈4 MB cada uno, ≈850 MB en total). Validación se usa en la sección 9 para elegir márgenes; prueba, para reportar.

Recordatorio de la sección 5.1: este Dice es optimista porque MedSAM probablemente vio parte de estas imágenes.

In [ ]:
val_ids = split.loc[split["split"] == "val", "image_id"].tolist()
test_ids = split.loc[split["split"] == "test", "image_id"].tolist()

cache = {}
start = time.time()
for image_id in tqdm(val_ids + test_ids, desc="Embeddings"):
    img, mask = load_pair(image_id)
    cache[image_id] = {"img": img, "mask": mask, "embedding": embed(img)}
print(f"{len(cache)} embeddings en {time.time() - start:.0f} s")


def evaluate(image_ids, boxes_fn):
    """Dice por imagen. boxes_fn(c) devuelve las cajas (K, 4) para la entrada c del caché."""
    rows = []
    for image_id in image_ids:
        c = cache[image_id]
        pred = to_mask(segment(c["embedding"], boxes_fn(c)))
        rows.append({"image_id": image_id, "dice": dice(pred, c["mask"])})
    return pd.DataFrame(rows).merge(split[["image_id", "source"]], on="image_id")


def oracle_boxes(c):
    return lung_boxes(c["mask"])


def single_box(c):
    return full_box(c["mask"])

### 8.1 Una caja por pulmón frente a una caja para ambos

Ambas son cajas oráculo (calculadas desde la máscara real). Solo cambia el número de cajas.

In [ ]:
res_two = evaluate(test_ids, oracle_boxes)
res_one = evaluate(test_ids, single_box)

summary = pd.DataFrame({
    "una caja por pulmón": res_two.groupby("source")["dice"].mean(),
    "una caja para ambos": res_one.groupby("source")["dice"].mean(),
})
summary.loc["total"] = [res_two["dice"].mean(), res_one["dice"].mean()]
print("Dice medio en test (cajas oráculo)")
display(summary.round(3))
display(res_two.groupby("source")["dice"].describe().round(3))

fig, ax = plt.subplots(figsize=(7, 3.2))
bins = np.linspace(min(res_two["dice"].min(), res_one["dice"].min()), 1, 30)
ax.hist(res_two["dice"], bins=bins, color=C_TRUE, alpha=0.8, label="una caja por pulmón")
ax.hist(res_one["dice"], bins=bins, color=C_PRED, alpha=0.6, label="una caja para ambos")
ax.set_title("Dice por imagen en test", loc="left"); ax.set_xlabel("Dice"); ax.set_ylabel("Imágenes")
ax.legend(frameon=False)
plt.show()

Como en U-Net, el promedio esconde los casos difíciles. Hay que mirar los peores: ahí se ve si el error está en los bordes, en los ápices, en los ángulos costofrénicos, en zonas opacas por enfermedad o en el mediastino (este último es típico de la caja única).

In [ ]:
def show_cases(results, boxes_fn, title):
    fig, axes = plt.subplots(1, len(results), figsize=(3.2 * len(results), 3.6))
    for ax, r in zip(np.atleast_1d(axes), results.itertuples()):
        c = cache[r.image_id]
        boxes = boxes_fn(c)
        draw(ax, c["img"], true=c["mask"], pred=to_mask(segment(c["embedding"], boxes)), boxes=boxes,
             title=f"{r.image_id} (Dice {r.dice:.2f})")
    fig.suptitle(title, x=0.01, ha="left")
    fig.tight_layout()
    plt.show()


ordered = res_two.sort_values("dice")
show_cases(ordered.head(4), oracle_boxes, "Una caja por pulmón, peores casos — azul: real, naranja: MedSAM, verde: caja")
show_cases(ordered.tail(4), oracle_boxes, "Una caja por pulmón, mejores casos")
show_cases(res_one.sort_values("dice").head(4), single_box, "Una caja para ambos, peores casos")

## 9. Sensibilidad a la caja y cajas automáticas

La caja oráculo no existe en TBX11K. Antes de reemplazarla hay que saber **cuánto depende el resultado de la precisión de la caja**.

### 9.1 Cajas más grandes que el pulmón

Se agranda la caja oráculo un margen fijo por lado y se mide el Dice en **validación**. MedSAM se entrenó con cajas perturbadas al azar hasta 20 píxeles por lado (`bbox_shift=20` en `train_one_gpu.py`), así que se espera cierta tolerancia a cajas imprecisas; la curva muestra cuánta.

In [ ]:
MARGINS = [0, 10, 20, 40, 60]
oracle_sweep = pd.Series(
    {m: evaluate(val_ids, lambda c, m=m: expand(oracle_boxes(c), m))["dice"].mean() for m in MARGINS},
    name="caja oráculo",
)
display(oracle_sweep.rename_axis("margen (px)").to_frame().round(3))

image_id = val_ids[0]
c = cache[image_id]
fig, axes = plt.subplots(1, len(MARGINS), figsize=(3 * len(MARGINS), 3.4))
for ax, m in zip(axes, MARGINS):
    boxes = expand(oracle_boxes(c), m)
    pred = to_mask(segment(c["embedding"], boxes))
    draw(ax, c["img"], true=c["mask"], pred=pred, boxes=boxes, title=f"margen {m} px — Dice {dice(pred, c['mask']):.3f}")
fig.suptitle(f"{image_id}: la misma imagen con cajas cada vez más grandes", x=0.01, ha="left")
fig.tight_layout()
plt.show()

### 9.2 Caja a priori: la caja típica de cada pulmón

La caja a priori es **la misma para todas las imágenes**: la mediana, coordenada por coordenada, de las cajas oráculo del conjunto de **entrenamiento**. Solo se calcula a partir de las máscaras (no hace falta el modelo), y después se aplica a imágenes sin máscara.

Es la idea que plantea el notebook de exploración (sección 3.5): derivar las cajas de una posición promedio, con un margen generoso porque la posición del pulmón varía entre radiografías. El margen se elige en validación.

In [ ]:
train_ids = split.loc[split["split"] == "train", "image_id"].tolist()
train_boxes, mean_mask = [], np.zeros((IMAGE_SIZE, IMAGE_SIZE))
for image_id in tqdm(train_ids, desc="Cajas de train"):
    mask = load_pair(image_id)[1]
    train_boxes.append(lung_boxes(mask))
    mean_mask += mask
train_boxes, mean_mask = np.stack(train_boxes), mean_mask / len(train_ids)   # (N, 2, 4)

prior = np.median(train_boxes, axis=0).round().astype(int)
sides = ["pulmón a la izquierda de la imagen", "pulmón a la derecha de la imagen"]
cols = ["x0", "y0", "x1", "y1"]
display(pd.concat({"caja a priori (mediana)": pd.DataFrame(prior, index=sides, columns=cols),
                   "desviación estándar": pd.DataFrame(train_boxes.std(axis=0), index=sides, columns=cols).round(1)},
                  axis=1))

fig, ax = plt.subplots(figsize=(4.2, 4.2))
ax.imshow(mean_mask, cmap="Blues", vmin=0, vmax=1)
for x0, y0, x1, y1 in train_boxes[::5].reshape(-1, 4):
    ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec="#8a8984", lw=0.4, alpha=0.5))
for x0, y0, x1, y1 in prior:
    ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec=C_BOX, lw=2))
ax.set_title("Máscara promedio de train, 1 de cada 5 cajas\n(gris) y caja a priori (verde)", fontsize=9)
ax.axis("off")
plt.show()

prior_sweep = pd.Series(
    {m: evaluate(val_ids, lambda c, m=m: expand(prior, m))["dice"].mean() for m in MARGINS},
    name="caja a priori",
)
prior_margin = int(prior_sweep.idxmax())
print(f"Margen elegido en validación para la caja a priori: {prior_margin} px")

### 9.3 Refinar en dos pasadas

La caja a priori es imprecisa por diseño. Como el embedding ya está calculado, una segunda pasada cuesta solo el decodificador:

1. **Primera pasada**: MedSAM segmenta con la caja a priori.
2. Para cada caja, se toma el componente conectado más grande de su máscara y se calcula su caja (+ un margen).
3. **Segunda pasada**: MedSAM segmenta otra vez con esas cajas, que ya se adaptan a la imagen.

Si la primera pasada no encuentra pulmón dentro de una caja, se conserva la caja a priori. El margen de la segunda pasada también se elige en validación.

In [ ]:
def largest_component(mask):
    """Componente conectado más grande de una máscara booleana (vacía si no hay ninguno)."""
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask.astype(np.uint8), connectivity=8)
    if n == 1:
        return np.zeros_like(mask, dtype=bool)
    return labels == 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])


def refine_boxes(embedding, boxes, margin):
    """Segunda pasada: la caja de cada máscara de la primera pasada (+ margen) reemplaza a la caja inicial."""
    new_boxes = []
    for box, p in zip(boxes, segment(embedding, boxes)):
        part = largest_component(p > 0.5)
        if part.any():
            ys, xs = np.nonzero(part)
            box = expand([xs.min(), ys.min(), xs.max(), ys.max()], margin)
        new_boxes.append(box)
    return np.array(new_boxes)


first_boxes = expand(prior, prior_margin)
REFINE_MARGINS = [0, 5, 10, 20]
refine_sweep = pd.Series(
    {m: evaluate(val_ids, lambda c, m=m: refine_boxes(c["embedding"], first_boxes, m))["dice"].mean()
     for m in REFINE_MARGINS},
    name="a priori + refinamiento",
)
refine_margin = int(refine_sweep.idxmax())
print(f"Margen elegido en validación para la segunda pasada: {refine_margin} px")

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(oracle_sweep.index, oracle_sweep.values, "o-", color=C_TRUE, lw=2, label="caja oráculo + margen")
ax.plot(prior_sweep.index, prior_sweep.values, "o-", color=C_BOX, lw=2, label="caja a priori + margen")
ax.plot(refine_sweep.index, refine_sweep.values, "o-", color=C_PRED, lw=2, label="a priori + refinamiento (margen de 2.ª pasada)")
ax.set_xlabel("Margen por lado (px, imagen de 512)"); ax.set_ylabel("Dice medio en validación")
ax.set_title("Sensibilidad a la caja", loc="left"); ax.grid(alpha=0.3); ax.legend(frameon=False, fontsize=9)
plt.show()

Cómo leer la figura:
- **Caja oráculo**: cuánto se pierde cuando la caja es más grande que el pulmón. Es el techo de cualquier estrategia automática.
- **Caja a priori**: un margen pequeño recorta los pulmones que se salen de la caja típica; uno grande incluye más estructuras vecinas. El mejor margen equilibra ambos errores.
- **Refinamiento**: si su curva queda por encima de la de la caja a priori, la segunda pasada corrige parte del error de la caja típica.

### 9.4 Resultado en test

Las tres estrategias se evalúan en test **una sola vez**, con los márgenes elegidos en validación.

In [ ]:
strategies = {
    "oráculo (una caja por pulmón)": oracle_boxes,
    f"a priori (margen {prior_margin})": lambda c: first_boxes,
    f"a priori + refinamiento (margen {refine_margin})": lambda c: refine_boxes(c["embedding"], first_boxes, refine_margin),
}
test_results = {name: evaluate(test_ids, fn) for name, fn in strategies.items()}
table = pd.DataFrame({name: r.groupby("source")["dice"].mean() for name, r in test_results.items()}).T
table["total"] = [r["dice"].mean() for r in test_results.values()]
table["mínimo"] = [r["dice"].min() for r in test_results.values()]
print("Dice medio en test")
display(table.round(3))

auto_name = list(strategies)[2]
show_cases(test_results[auto_name].sort_values("dice").head(4), strategies[auto_name],
           "A priori + refinamiento, peores casos en test")

## 10. Aplicación a TBX11K

Este es el uso real en el proyecto: segmentar radiografías **sin máscara** con cajas automáticas. No se puede calcular el Dice, así que la revisión es **visual**.

### 10.1 ¿La caja a priori sirve para TBX11K?

La caja a priori se aprendió de Montgomery + Shenzhen. Si TBX11K encuadra el tórax de otra manera, la caja queda desplazada. Se compara con la imagen promedio de las 1600 radiografías del subconjunto del proyecto. En azul, el contorno de la máscara promedio de Montgomery + Shenzhen (píxeles que son pulmón en al menos la mitad de las imágenes de train).

In [ ]:
subset = load_subset()
mean_tbx = np.zeros((IMAGE_SIZE, IMAGE_SIZE))
for rel_path in tqdm(subset["rel_path"], desc="TBX11K"):
    mean_tbx += load_gray(rel_path)
mean_tbx /= len(subset)

fig, ax = plt.subplots(figsize=(4.2, 4.2))
draw(ax, mean_tbx, true=mean_mask >= 0.5, boxes=first_boxes,
     title=f"Promedio de TBX11K ({len(subset)} imágenes)\nazul: pulmón típico M+S, verde: caja a priori + {prior_margin} px")
plt.show()

### 10.2 Segmentación

Seis radiografías del subconjunto, dos por clase. Fila superior: solo la caja a priori. Fila inferior: a priori + refinamiento.

In [ ]:
sample = subset.groupby("source_class").sample(2, random_state=SEED)
fig, axes = plt.subplots(2, len(sample), figsize=(3 * len(sample), 6.6))
for col, r in enumerate(sample.itertuples()):
    img = load_gray(r.rel_path)
    embedding = embed(img)
    refined = refine_boxes(embedding, first_boxes, refine_margin)
    for row, boxes in enumerate([first_boxes, refined]):
        draw(axes[row, col], img, pred=to_mask(segment(embedding, boxes)), boxes=boxes,
             title=f"{r.image_id} ({r.source_class})")
axes[0, 0].text(-30, 256, "A priori", rotation=90, va="center", ha="right")
axes[1, 0].text(-30, 256, "A priori + refinamiento", rotation=90, va="center", ha="right")
fig.suptitle("TBX11K — naranja: MedSAM, verde: caja (sin máscara de referencia)", x=0.01, ha="left")
fig.tight_layout()
plt.show()

Qué revisar:
- **Pulmón que sobresale de la caja**: la caja es la única indicación que recibe el modelo sobre dónde está el pulmón. Revisar sobre todo los **ápices**, donde se concentran las lesiones de TB (notebook de exploración).
- **Zonas opacas por enfermedad** que el modelo excluye del pulmón.
- **Corazón o mediastino** dentro de la máscara cuando la caja es muy ancha.
- **Fragmentos sueltos** dentro de la caja pero fuera del pulmón.

Son los mismos errores que se buscan en U-Net y los que revisará el notebook de control de calidad. El posprocesamiento (quedarse con el componente más grande por caja y rellenar huecos) corrige parte de ellos.

## 11. Qué hay que tener en cuenta

**Datos**
- **Contaminación**: MedSAM se desarrolló con Montgomery + Shenzhen (sección 5.1). Ningún Dice calculado en ese dataset sirve para comparar MedSAM con U-Net o con otros métodos.
- **Mismo preprocesamiento que en el entrenamiento**: recorte de percentiles, 1024×1024, [0, 1], 3 canales. Usar `SamPredictor` o la normalización de SAM degrada la predicción sin dar error.

**Prompt**
- **El resultado depende de la caja.** MedSAM no "sabe" que busca pulmón: segmenta lo que la caja delimita. Con una caja mala, la máscara es mala aunque el modelo sea bueno.
- **Una estructura por caja.** Una caja por pulmón; la máscara final es la unión.
- **Los márgenes se eligen en validación**, nunca en test, por la misma razón que en U-Net.
- **Coordenadas**: `[x0, y0, x1, y1]` en píxeles de la imagen original, reescaladas a 1024 antes de entrar al modelo. Un error de escala desplaza la caja sin dar error.

**Inferencia**
- `medsam.eval()` y `torch.no_grad()` siempre.
- **El umbral 0,5 es una convención**, como en U-Net.
- **Resolución de salida**: el decodificador produce 256×256 y se reescala a 512. El detalle del borde queda limitado por esa resolución, aunque la entrada sea de 1024.
- **Cómputo**: el codificador de imagen es la parte cara (0,8–3,4 s por imagen en una GTX 1650 Ti, ≈17 s en CPU). Calcular el embedding una vez y reutilizarlo con todas las cajas.
- **Determinismo**: no hay entrenamiento ni aleatoriedad en la inferencia; la misma imagen con la misma caja da la misma máscara, salvo diferencias numéricas menores entre GPU y CPU.

**Licencia**: MedSAM y `segment-anything` se distribuyen con licencia Apache 2.0.

## 12. Ajuste fino (*fine-tuning*)

MedSAM se puede seguir entrenando con datos propios (`train_one_gpu.py` del repositorio): misma pérdida Dice + entropía cruzada, AdamW con tasa de aprendizaje 1e-4, codificador de prompt congelado y cajas obtenidas de la máscara real con una perturbación aleatoria de hasta 20 píxeles.

**Por qué no se hace en este tutorial**
- **Los datos con máscara del proyecto ya están en el entrenamiento de MedSAM** (sección 5.1). Ajustar con ellos aporta poco y no resuelve la contaminación.
- **Memoria**: el ajuste completo actualiza ≈93,7 M parámetros. Solo pesos, gradientes y los dos estados de AdamW ocupan ≈93,7 M × 16 bytes ≈ 1,5 GB, sin contar las activaciones del codificador a 1024×1024. La inferencia sola ya usa 2,8 GB, así que el ajuste completo no cabe en una GPU de 4 GB.

**Alternativa liviana, si hiciera falta** (por ejemplo, con un dataset de pulmón que MedSAM no haya visto): congelar el codificador de imagen, calcular los embeddings una vez, como en la sección 8, y entrenar solo el decodificador de máscara (4,1 M parámetros):

```python
for p in medsam.image_encoder.parameters():
    p.requires_grad = False
for p in medsam.prompt_encoder.parameters():
    p.requires_grad = False
optimizer = torch.optim.AdamW(medsam.mask_decoder.parameters(), lr=1e-4, weight_decay=0.01)
# Por lote: embeddings guardados + caja oráculo con perturbación aleatoria
#   -> decodificador -> logits 256x256 -> reescalar -> pérdida Dice + BCE (tutorial de U-Net)
```

Los embeddings de train ocupan ≈4 MB cada uno (≈2 GB para 492 imágenes).

## 13. Relación con el proyecto

| Tutorial | Proyecto |
|---|---|
| Pesos en `models/medsam/medsam_vit_b.pth` | Mismo lugar (`models/` no se versiona) |
| `torchvision` y `segment-anything` instalados a mano | Agregarlos a `requirements.txt` |
| Cajas a priori elegidas en validación de Montgomery + Shenzhen | Mismas cajas y márgenes; o cajas desde la máscara de U-Net |
| Revisión visual de 6 imágenes de TBX11K | Predicción de las 1600 imágenes del subconjunto (entre 20 y 90 min en una GTX 1650 Ti) en `masks/medsam/` + control de calidad |
| Dice en Montgomery + Shenzhen (optimista) | No usarlo para comparar métodos |

**MedSAM frente a U-Net en el proyecto**
- U-Net aprende *qué* es pulmón a partir de las máscaras. MedSAM recibe *dónde* está el pulmón (la caja) y delimita el borde.
- Las dos estrategias se pueden combinar: la máscara de U-Net da la caja y MedSAM refina el borde.
- Para la pregunta del proyecto (efecto de la segmentación en las características radiómicas) importa el borde, y este tutorial muestra que el borde de MedSAM depende de la caja. La estrategia de caja forma parte del método y hay que reportarla.

## Referencias

- Ma, J., He, Y., Li, F., Han, L., You, C., & Wang, B. (2024). Segment anything in medical images. *Nature Communications*, 15, 654. https://doi.org/10.1038/s41467-024-44824-z
- Kirillov, A., et al. (2023). Segment Anything. *ICCV*, 4015–4026.
- Jaeger, S., et al. (2014). Two public chest X-ray datasets for computer-aided screening of pulmonary diseases. *Quant. Imaging Med. Surg.*, 4(6), 475–477.
- Repositorio de MedSAM: https://github.com/bowang-lab/MedSAM (`MedSAM_Inference.py`, `train_one_gpu.py`).
- Repositorio de SAM: https://github.com/facebookresearch/segment-anything